# FXS / 03c — Cost sensitivity of the 03b portfolio: spread 0–3× Dukascopy, commission $0–8 per lot

Same portfolio and period as 03b: EURUSD-02 + USDJPY-01 + USDJPY-02, every trading day, 2021-01 → 2026-11-06, $6,000 start, size = L × current equity per structure (compounding).

For **each leverage** (1, 2, 3, 5, 10× per structure) a table of the **annual return (CAGR, %)** over:
- **rows:** spread as a multiple of the Dukascopy quoted spread (0 = filled at the mid, 1 = Dukascopy, 3 = three times wider);
- **columns:** commission in USD per 100,000-unit lot, round trip ($0 → $8).

A second set of tables gives the maximum drawdown. Reference points: Dukascopy spread ≈ 0.3 pips (EURUSD) and ≈ 0.3–0.6 pips (USDJPY); raw/ECN brokers $6–7 per lot round trip; FundedNext about $3.

Descriptive, like 03b: the structures were chosen with 2021–2025 results, and 2026 has already been seen.

In [ ]:
import json, os, sys, tempfile
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

E0 = 6_000.0
LEVERAGES = [1, 2, 3, 5, 10]
SPREAD_FACTORS = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0]
COMMISSIONS = [0, 1, 2, 3, 4, 5, 6, 7, 8]          # USD per lot, round trip
IDS = ["EURUSD-02", "USDJPY-01", "USDJPY-02"]
FROZEN = {"EURUSD-02": ("NY", 660, 900, -1), "USDJPY-01": ("UTC", 55, 115, -1), "USDJPY-02": ("UTC", 25, 55, 1)}
WINDOWS = ((dt.date(2021, 1, 1), dt.date(2025, 12, 31), "confirmation"), (dt.date(2026, 1, 1), dt.date(2026, 11, 6), "holdout"))

SYNTHETIC = os.environ.get("FXS_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("FXS_SYNTH_DIR") or tempfile.mkdtemp(prefix="fxs03c_")
    os.environ.pop("DUKASCOPY_DIR", None)
    LEVERAGES, SPREAD_FACTORS, COMMISSIONS = [1, 5], [0.0, 1.0, 3.0], [0, 3, 8]
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F

cfg = F.FXSConfig()
BRANCH = Path.cwd().parent
OUT = Path.cwd() / ("03c_synthetic" if SYNTHETIC else "03c"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220, "display.max_columns", 40)

## 0. Synthetic inputs (smoke run only)

In [ ]:
if SYNTHETIC:
    base = Path(os.environ["EQUITY_TRADING_DIR"]) / "Dukascopy"
    for i, p in enumerate(("EURUSD", "USDJPY")):
        F.write_synthetic_candles(base, p, dt.date(2020, 12, 28), dt.date(2026, 11, 6), seed=7 + i, drift_bps=-3.0 if p == "EURUSD" else 0.0,
                                  drift_ny=(4, 8), price=150.0 if p.endswith("JPY") else 1.10)
    cand = pd.DataFrame([{"id": k, "pair": k[:6], "clock": c, "start": s, "end": e, "direction": d} for k, (c, s, e, d) in FROZEN.items()])
else:
    cand = F.load_frozen_candidates(BRANCH).query("id in @IDS").reset_index(drop=True)
for _, r in cand.iterrows():
    assert (r["clock"], int(r["start"]), int(r["end"]), int(r["direction"])) == FROZEN[r["id"]], f"{r['id']} is not the frozen rule"
PAIR_OF = cand.set_index("id")["pair"]

## 1. Returns at the mid and the cost of one Dukascopy spread, per trade

Any spread multiple `s` is then `mid return − s × Dukascopy spread`, and any commission `c` is `c / (100,000 × base-currency value)`.

In [ ]:
def detail(sf):
    parts = [F.structure_detail(cand, a, b, st, cfg, with_price=True, spread_factor=sf) for a, b, st in WINDOWS]
    return [pd.concat([x[k] for x in parts]).sort_index()[IDS] for k in range(3)]
for p in sorted(set(PAIR_OF)):
    for a, b, _ in WINDOWS:
        cov = F.file_coverage(p, a, b)
        assert cov["share"] >= cfg.min_file_share, f"{p} {a.year}: only {cov['share']:.1%} of day files"
G_mid, M_mid, P = detail(0.0)
G_one, M_one, _ = detail(1.0)
G_mid, G_one, M_mid, M_one = (x.fillna(0.0) for x in (G_mid, G_one, M_mid, M_one))
P = P.ffill().bfill()
SPREAD = G_mid - G_one                                          # one Dukascopy spread per trade (>= 0), bps
traded = G_one != 0
pip = lambda X, i: F.bps_to_pips(X[i], P[i], PAIR_OF[i])
ref = pd.DataFrame({"trades": traded.sum(),
                    "mid-to-mid pips/trade": pd.Series({i: pip(G_mid, i)[traded[i]].mean() for i in IDS}),
                    "Dukascopy spread, pips/trade": pd.Series({i: pip(SPREAD, i)[traded[i]].mean() for i in IDS}),
                    "$1/lot commission, pips": pd.Series({i: F.bps_to_pips(F.commission_bps_per_lot(PAIR_OF[i], 1.0), P[i][traded[i]], PAIR_OF[i]).mean() for i in IDS})})
display(ref.round(3)); ref.to_csv(OUT / "03c_reference.csv")

## 2. The grid

For each leverage, spread multiple and commission: the compounding account from $6,000, 2021-01 → 2026-11.

In [ ]:
comm1 = pd.Series({i: F.commission_bps_per_lot(PAIR_OF[i], 1.0) for i in IDS})
rows = []
for L in LEVERAGES:
    w = pd.Series(float(L), index=IDS)
    for sf in SPREAD_FACTORS:
        for c in COMMISSIONS:
            N = G_mid - sf * SPREAD - c * comm1
            st = F.account_stats(F.simulate_account(N, w, E0), E0)
            rows.append({"L": L, "spread_x": sf, "commission": c, "cagr": st["cagr"], "final_equity": st["final_equity"],
                         "max_drawdown": st["max_drawdown"], "worst_year": st["worst_year"], "ruined": st["ruined"]})
grid = pd.DataFrame(rows)
grid.to_csv(OUT / "03c_grid.csv", index=False)
print(len(grid), "accounts simulated")

## 3. Annual return (CAGR, %) per leverage

Rows: spread × Dukascopy. Columns: commission $ per lot round trip. The black box marks Dukascopy spread at $3 per lot.

In [ ]:
def table(L, col):
    return grid[grid["L"] == L].pivot(index="spread_x", columns="commission", values=col)

for L in LEVERAGES:
    t = 100 * table(L, "cagr")
    print(f"\n=== {L}x per structure: annual return, % ===")
    display(t.round(1).style.background_gradient(cmap="RdYlGn", vmin=-30, vmax=30, axis=None).format("{:.1f}"))
    t.to_csv(OUT / f"03c_cagr_L{L}.csv")

fig, axes = plt.subplots(1, len(LEVERAGES), figsize=(4.2 * len(LEVERAGES), 4), sharey=True)
for ax, L in zip(np.atleast_1d(axes), LEVERAGES):
    t = 100 * table(L, "cagr")
    lim = max(5.0, float(np.nanmax(np.abs(t.to_numpy()))))
    im = ax.imshow(t.to_numpy(), cmap="RdYlGn", vmin=-lim, vmax=lim, aspect="auto", origin="lower")
    ax.contour(t.to_numpy(), levels=[0], colors="black", linewidths=1.2, origin="lower")
    ax.set_xticks(range(len(t.columns))); ax.set_xticklabels([f"${c}" for c in t.columns], fontsize=8)
    ax.set_yticks(range(len(t.index))); ax.set_yticklabels([f"{s:g}x" for s in t.index])
    if 1.0 in list(t.index) and 3 in list(t.columns):
        y, x = list(t.index).index(1.0), list(t.columns).index(3)
        ax.add_patch(plt.Rectangle((x - 0.5, y - 0.5), 1, 1, fill=False, ec="black", lw=2))
    ax.set_title(f"{L}x each: CAGR %"); ax.set_xlabel("commission per lot")
    fig.colorbar(im, ax=ax, fraction=0.045)
np.atleast_1d(axes)[0].set_ylabel("spread x Dukascopy")
plt.tight_layout(); plt.savefig(OUT / "03c_cagr_heatmaps.png", dpi=110); plt.show()

## 4. Maximum drawdown (%) per leverage

In [ ]:
for L in LEVERAGES:
    t = 100 * table(L, "max_drawdown")
    print(f"\n=== {L}x per structure: maximum drawdown, % ===")
    display(t.round(1).style.background_gradient(cmap="RdYlGn", vmin=-60, vmax=0, axis=None).format("{:.1f}"))
    t.to_csv(OUT / f"03c_maxdd_L{L}.csv")

## 5. Break-even: the highest commission that still leaves a positive return, at each spread multiple

In [ ]:
be = {}
for L in LEVERAGES:
    t = table(L, "cagr")
    be[f"{L}x"] = pd.Series({sf: (max([c for c in t.columns if t.loc[sf, c] > 0]) if (t.loc[sf] > 0).any() else np.nan) for sf in t.index})
be = pd.DataFrame(be); be.index.name = "spread x Dukascopy"
display(be); be.to_csv(OUT / "03c_breakeven_commission.csv")
print("NaN = negative at every commission tested; the top of the range ($%d) means positive even at the highest commission." % max(COMMISSIONS))

In [ ]:
summary = {"study": "FXS03c", "descriptive": True, "synthetic": SYNTHETIC, "structures": IDS, "equity0": E0,
           "leverages": LEVERAGES, "spread_factors": SPREAD_FACTORS, "commissions_usd_per_lot_rt": COMMISSIONS,
           "reference": ref.round(4).reset_index().to_dict(orient="records"),
           "breakeven_commission": be.reset_index().to_dict(orient="records"),
           "note": "Same portfolio as 03b; hindsight selection; 2026 already seen in FXS 05/06."}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("written", OUT / "summary.json")